In [1]:
import pandas as pd
import csv
import numpy as np

from statsmodels.tsa.arima.model import ARIMA
from pmdarima import auto_arima
from statsmodels.tsa.statespace.sarimax import SARIMAX

from sklearn.model_selection import GridSearchCV
from sklearn.metrics import mean_squared_error

import matplotlib.pyplot as plt

In [3]:
# Read in PA data and re-format date to the correct format
PA = pd.read_csv('Data/Pennsylvania.csv', delimiter=",")
PA.rename(columns={'date': 'time'}, inplace=True)
PA['time'] = pd.to_datetime(PA['time'])
PA['date'] = PA['time'].dt.date
display(PA.head(3))

# Agg VADAR score based on business id and dates
agg_PA=PA.groupby(['business_id', 'date'])['doc_sentiment'].agg(avg_VADAR='mean').reset_index()
agg_PA = agg_PA.merge(PA, on=['business_id', 'date'])[['business_id','business_name','city','state','date','avg_VADAR']]
display(agg_PA.head(3))

print('Total Biz IDs:',len(agg_PA['business_id'].unique()))

,business_id,business_name,city,state,latitude,longitude,stars,total_review_count,review_id,rating,time,review,doc_sentiment,aspect_sentiments,date
0,PP3BBaVxZLcJU54uP_wL6Q,Pat's King of Steaks,Philadelphia,PA,39.933201,-75.159266,3.0,4250,g80vzN72iU03Wh0fSpq41g,5.0,2005-02-16 04:06:26,These guys really are the king of cheese steak...,0.0000,"{'PRODUCT': {}, 'PERSON': {}, 'ORG': {}}",2005-02-16
1,Co3Ogqy6y2JgZdG0wBlrUQ,Ten Stone Bar & Restaurant,Philadelphia,PA,39.945046,-75.176973,3.0,300,DTrvaOwqev-xhbqqblt7Tw,5.0,2005-05-25 01:12:54,THIS IS MY FAVORITE BAR IN PHILADELPHIA. Oh T...,0.9736,"{'PRODUCT': {}, 'PERSON': {'Ten Stone': -0.22}...",2005-05-25
2,RQAF6a0akMiot5lZZnMNNw,Dalessandro’s Steaks & Hoagies,Philadelphia,PA,40.029494,-75.205971,4.0,2686,J3Jk5A1TnFeKf2SGNrZLrw,5.0,2005-05-26 04:09:08,I lost a bet during undergrad at Carnegie Mell...,0.6124,"{'PRODUCT': {}, 'PERSON': {}, 'ORG': {'Carnegi...",2005-05-26


,business_id,business_name,city,state,date,avg_VADAR
0,--30_8IhuyMHbSOcNWd6DQ,Action Karate,Jamison,PA,2012-07-18,0.7463
1,--30_8IhuyMHbSOcNWd6DQ,Action Karate,Jamison,PA,2013-01-31,0.9381
2,--30_8IhuyMHbSOcNWd6DQ,Action Karate,Jamison,PA,2013-04-20,-0.8837


Total Biz IDs: 34039


In [74]:
####### Run all PA business ############

#Note: please ignore the incorrectly labeled name, such as 'random_xxx'. 
      #They are directly inherited from the test run

n=len(agg_PA['business_id'].unique())
random_business_ids = agg_PA['business_id'].unique()
random_n = agg_PA.copy()
random_n['date_index'] = pd.to_datetime(random_n['date'])
random_n.set_index('date_index', inplace=True)

random_n_vadar=random_n.copy()[['business_id','avg_VADAR','business_name']]
business_ids = random_n_vadar['business_id'].unique()
sample_freq='6M'

# Initialize a variable to keep track of overall accuracy.
overall_accuracy = []

test_predictions_df = pd.DataFrame()
predictions_df=pd.DataFrame()

# Group data by index
for i,business_id in enumerate(business_ids):

    business_data = random_n_vadar[random_n_vadar['business_id'] == business_id]
    
    # Group data by the half_year period and calculate the mean to reduce noise
    HalfYear_grouped = business_data.resample(sample_freq).mean(numeric_only=True)

    #Fill NaN values with the average of the previous and following values if there are still NaN
    HalfYear_grouped_filled = HalfYear_grouped.fillna(method='ffill').fillna(method='bfill')
    
    # Skip the business_id if there are NaNs or too few records to run ARIMA
    if HalfYear_grouped_filled['avg_VADAR'].isna().all() or len(HalfYear_grouped_filled)<5:
        continue

    # Calculate the index for the 80% train-test split.
    split_index = int(0.8 * len(HalfYear_grouped_filled ))
    
    # Split the data into training and testing sets.
    train_data = HalfYear_grouped_filled.iloc[:split_index]
    test_data = HalfYear_grouped_filled.iloc[split_index:]

    # Use auto_arima to optimize both trend and order
    trend_values = ['n', 'c', 't', 'ct']
    best_mse=float('inf')
    for trend_value in trend_values:
        best_model = auto_arima(
            train_data,
            seasonal=True,
            stepwise=True,
            suppress_warnings=True,
            trend=trend_value,
            error_action='ignore')

        # Make predictions on the test data using the best model
        test_predictions = best_model.predict(n_periods=len(test_data))

        # Calculate the MSE for hyperparameters selection
        mse_current = mean_squared_error(test_data['avg_VADAR'], test_predictions)

        if mse_current< best_mse:
            best_mse = mse_current
            best_trend = trend_value
            best_order=best_model.get_params()['order']

    print('best trend',best_trend)
    print('best order',best_order)
    print('-------------------------')

    test_predictions_data = pd.DataFrame({
        'Date': test_data.index,
        'Business_ID': business_id,
        'Actual_VADAR': test_data['avg_VADAR'],
        'Predicted_VADAR': test_predictions
    })
    test_predictions_df = pd.concat([test_predictions_df, test_predictions_data], ignore_index=True)

    # Append the accuracy to the overall_accuracy list.
    overall_accuracy.append(best_mse)

    # Make prediction based on all data
    best_model = auto_arima(
            HalfYear_grouped_filled,
            seasonal=True,
            stepwise=True,
            suppress_warnings=True,
            trend=best_trend,
            error_action='ignore')
    predictions = best_model.predict(n_periods=1)
    
    predictions_data = pd.DataFrame({
        'Business_ID': business_id,
        'Date': HalfYear_grouped_filled.index[-1]+ pd.DateOffset(months=6),
        'Predicted_VADAR': predictions
    })
    predictions_df = pd.concat([predictions_df, predictions_data], ignore_index=True)

# Calculate the average accuracy across all businesses.
average_accuracy = np.mean(overall_accuracy)
print(f"Average Mean Squared Error (MSE) for {n} businesses: {average_accuracy}")

display(test_predictions_df.head(5))
display(predictions_df.head(5))


best trend c
best order (1, 0, 0)
-------------------------
best trend n
best order (1, 0, 0)
-------------------------
best trend n
best order (1, 0, 0)
-------------------------
best trend n
best order (1, 0, 0)
-------------------------
best trend n
best order (0, 0, 0)
-------------------------
best trend n
best order (0, 1, 0)
-------------------------
best trend c
best order (0, 0, 0)
-------------------------
best trend n
best order (1, 0, 0)
-------------------------
best trend c
best order (3, 0, 1)
-------------------------
best trend c
best order (0, 0, 1)
-------------------------
best trend c
best order (0, 0, 0)
-------------------------
best trend t
best order (1, 0, 0)
-------------------------
best trend c
best order (0, 0, 1)
-------------------------
best trend t
best order (1, 0, 1)
-------------------------
best trend n
best order (1, 0, 1)
-------------------------
best trend n
best order (1, 0, 1)
-------------------------
best trend c
best order (1, 0, 0)
------

/Users/enhongliu/anaconda3/lib/python3.11/site-packages/pmdarima/arima/auto.py:444: UserWarning: Input time-series is completely constant; returning a (0, 0, 0) ARMA.
  warnings.warn('Input time-series is completely constant; '
/Users/enhongliu/anaconda3/lib/python3.11/site-packages/pmdarima/arima/auto.py:444: UserWarning: Input time-series is completely constant; returning a (0, 0, 0) ARMA.
  warnings.warn('Input time-series is completely constant; '
/Users/enhongliu/anaconda3/lib/python3.11/site-packages/pmdarima/arima/auto.py:444: UserWarning: Input time-series is completely constant; returning a (0, 0, 0) ARMA.
  warnings.warn('Input time-series is completely constant; '
/Users/enhongliu/anaconda3/lib/python3.11/site-packages/pmdarima/arima/auto.py:444: UserWarning: Input time-series is completely constant; returning a (0, 0, 0) ARMA.
  warnings.warn('Input time-series is completely constant; '


best trend n
best order (0, 1, 1)
-------------------------
best trend n
best order (1, 0, 0)
-------------------------
best trend t
best order (2, 0, 1)
-------------------------
best trend n
best order (0, 0, 0)
-------------------------
best trend n
best order (0, 1, 0)
-------------------------
best trend c
best order (1, 0, 0)
-------------------------
best trend n
best order (0, 1, 1)
-------------------------
best trend t
best order (0, 0, 0)
-------------------------
best trend n
best order (1, 0, 1)
-------------------------
best trend ct
best order (0, 0, 1)
-------------------------
best trend ct
best order (0, 0, 0)
-------------------------
best trend t
best order (2, 0, 2)
-------------------------
best trend n
best order (1, 0, 0)
-------------------------
best trend ct
best order (0, 0, 0)
-------------------------
best trend c
best order (0, 0, 0)
-------------------------
best trend n
best order (1, 0, 0)
-------------------------
best trend n
best order (0, 0, 0)
---

/Users/enhongliu/anaconda3/lib/python3.11/site-packages/pmdarima/arima/auto.py:444: UserWarning: Input time-series is completely constant; returning a (0, 0, 0) ARMA.
  warnings.warn('Input time-series is completely constant; '
/Users/enhongliu/anaconda3/lib/python3.11/site-packages/pmdarima/arima/auto.py:444: UserWarning: Input time-series is completely constant; returning a (0, 0, 0) ARMA.
  warnings.warn('Input time-series is completely constant; '
/Users/enhongliu/anaconda3/lib/python3.11/site-packages/pmdarima/arima/auto.py:444: UserWarning: Input time-series is completely constant; returning a (0, 0, 0) ARMA.
  warnings.warn('Input time-series is completely constant; '
/Users/enhongliu/anaconda3/lib/python3.11/site-packages/pmdarima/arima/auto.py:444: UserWarning: Input time-series is completely constant; returning a (0, 0, 0) ARMA.
  warnings.warn('Input time-series is completely constant; '


best trend ct
best order (0, 0, 0)
-------------------------
best trend ct
best order (0, 0, 1)
-------------------------
best trend c
best order (0, 0, 0)
-------------------------
best trend c
best order (1, 0, 0)
-------------------------
best trend t
best order (0, 0, 0)
-------------------------
best trend ct
best order (1, 0, 1)
-------------------------
best trend t
best order (0, 0, 1)
-------------------------
best trend n
best order (1, 0, 0)
-------------------------
best trend n
best order (2, 0, 0)
-------------------------
best trend n
best order (1, 0, 1)
-------------------------
best trend c
best order (1, 0, 0)
-------------------------
best trend ct
best order (0, 0, 0)
-------------------------
best trend c
best order (0, 0, 0)
-------------------------
best trend c
best order (0, 0, 0)
-------------------------
best trend n
best order (1, 0, 0)
-------------------------
best trend c
best order (0, 0, 1)
-------------------------
best trend t
best order (1, 0, 0)
--

...

best trend ct
best order (0, 0, 0)
-------------------------
best trend c
best order (1, 0, 0)
-------------------------
best trend c
best order (0, 0, 1)
-------------------------
best trend ct
best order (0, 0, 0)
-------------------------


/Users/enhongliu/anaconda3/lib/python3.11/site-packages/pmdarima/arima/auto.py:444: UserWarning: Input time-series is completely constant; returning a (0, 0, 0) ARMA.
  warnings.warn('Input time-series is completely constant; '
/Users/enhongliu/anaconda3/lib/python3.11/site-packages/pmdarima/arima/auto.py:444: UserWarning: Input time-series is completely constant; returning a (0, 0, 0) ARMA.
  warnings.warn('Input time-series is completely constant; '
/Users/enhongliu/anaconda3/lib/python3.11/site-packages/pmdarima/arima/auto.py:444: UserWarning: Input time-series is completely constant; returning a (0, 0, 0) ARMA.
  warnings.warn('Input time-series is completely constant; '
/Users/enhongliu/anaconda3/lib/python3.11/site-packages/pmdarima/arima/auto.py:444: UserWarning: Input time-series is completely constant; returning a (0, 0, 0) ARMA.
  warnings.warn('Input time-series is completely constant; '


best trend c
best order (0, 0, 0)
-------------------------
best trend t
best order (0, 0, 0)
-------------------------
best trend ct
best order (0, 0, 1)
-------------------------
best trend ct
best order (0, 1, 1)
-------------------------
best trend ct
best order (1, 0, 0)
-------------------------
best trend n
best order (0, 1, 1)
-------------------------
best trend t
best order (1, 0, 1)
-------------------------
best trend t
best order (1, 0, 0)
-------------------------
best trend n
best order (1, 0, 0)
-------------------------
best trend c
best order (0, 0, 0)
-------------------------
best trend ct
best order (0, 0, 0)
-------------------------
best trend c
best order (0, 0, 0)
-------------------------
best trend ct
best order (0, 0, 0)
-------------------------
best trend ct
best order (1, 0, 0)
-------------------------
best trend n
best order (0, 0, 0)
-------------------------
best trend ct
best order (0, 0, 1)
-------------------------
best trend ct
best order (1, 0, 0

,Date,Business_ID,Actual_VADAR,Predicted_VADAR
0,2018-07-31,--30_8IhuyMHbSOcNWd6DQ,0.8857,-0.758637
1,2019-01-31,--30_8IhuyMHbSOcNWd6DQ,0.8857,-0.968922
2,2019-07-31,--30_8IhuyMHbSOcNWd6DQ,0.8857,-1.405817
3,2020-01-31,--30_8IhuyMHbSOcNWd6DQ,0.9951,-1.097393
4,2019-01-31,--OS_I7dnABrXvRCCuWOGQ,-0.8887,-1.419854


,Business_ID,Date,Predicted_VADAR
0,--30_8IhuyMHbSOcNWd6DQ,2020-07-31,0.760785
1,--OS_I7dnABrXvRCCuWOGQ,2020-01-31,0.000000
2,--ZVrH2X2QXBFdCilbirsw,2018-08-28,0.232407
3,--ZWv8kGlM2YL58uKhGJDg,2016-07-31,-0.384363
4,--epgcb7xHGuJ-4PUeSLAw,2022-10-30,0.099950


In [ ]:
# ######## Run 1000 random business to check coding bugs#######

# # List of unique business IDs.
# n=1000
# random_business_ids = agg_PA['business_id'].sample(n=n).unique()
# random_n = agg_PA[agg_PA['business_id'].isin(random_business_ids)].copy()
# random_n['date_index'] = pd.to_datetime(random_n['date'])
# random_n.set_index('date_index', inplace=True)

# random_n_vadar=random_n.copy()[['business_id','avg_VADAR','business_name']]
# business_ids = random_n_vadar['business_id'].unique()
# sample_freq='6M'

# # Initialize a variable to keep track of overall accuracy.
# overall_accuracy = []

# test_predictions_df = pd.DataFrame()
# predictions_df=pd.DataFrame()

# # Group data by index
# for i,business_id in enumerate(business_ids):

#     business_data = random_n_vadar[random_n_vadar['business_id'] == business_id]
    
#     # Group data by the half_year period and calculate the mean to reduce noise
#     HalfYear_grouped = business_data.resample(sample_freq).mean(numeric_only=True)

#     #Fill NaN values with the average of the previous and following values if there are still NaN
#     HalfYear_grouped_filled = HalfYear_grouped.fillna(method='ffill').fillna(method='bfill')
    
#     # Skip the business_id if there are NaNs or too few records to run ARIMA
#     if HalfYear_grouped_filled['avg_VADAR'].isna().all() or len(HalfYear_grouped_filled)<5:
#         continue

#     # Calculate the index for the 80% train-test split.
#     split_index = int(0.8 * len(HalfYear_grouped_filled ))
    
#     # Split the data into training and testing sets.
#     train_data = HalfYear_grouped_filled.iloc[:split_index]
#     test_data = HalfYear_grouped_filled.iloc[split_index:]

#     # Use auto_arima to optimize both trend and order
#     trend_values = ['n', 'c', 't', 'ct']
#     best_mse=float('inf')
#     for trend_value in trend_values:
#         best_model = auto_arima(
#             train_data,
#             seasonal=True,
#             stepwise=True,
#             suppress_warnings=True,
#             trend=trend_value,
#             error_action='ignore')

#         # Make predictions on the test data using the best model
#         test_predictions = best_model.predict(n_periods=len(test_data))

#         # Calculate the MSE for hyperparameters selection
#         mse_current = mean_squared_error(test_data['avg_VADAR'], test_predictions)

#         if mse_current< best_mse:
#             best_mse = mse_current
#             best_trend = trend_value
#             best_order=best_model.get_params()['order']

#     print('best trend',best_trend)
#     print('best order',best_order)
#     print('-------------------------')

#     test_predictions_data = pd.DataFrame({
#         'Date': test_data.index,
#         'Business_ID': business_id,
#         'Actual_VADAR': test_data['avg_VADAR'],
#         'Predicted_VADAR': test_predictions
#     })
#     test_predictions_df = pd.concat([test_predictions_df, test_predictions_data], ignore_index=True)

#     # Append the accuracy to the overall_accuracy list.
#     overall_accuracy.append(best_mse)

#     # Make prediction based on all data
#     best_model = auto_arima(
#             HalfYear_grouped_filled,
#             seasonal=True,
#             stepwise=True,
#             suppress_warnings=True,
#             trend=best_trend,
#             error_action='ignore')
#     predictions = best_model.predict(n_periods=1)
    
#     predictions_data = pd.DataFrame({
#         'Business_ID': business_id,
#         'Date': HalfYear_grouped_filled.index[-1]+ pd.DateOffset(months=6),
#         'Predicted_VADAR': predictions
#     })
#     predictions_df = pd.concat([predictions_df, predictions_data], ignore_index=True)

# # Calculate the average accuracy across all businesses.
# average_accuracy = np.mean(overall_accuracy)
# print(f"Average Mean Squared Error (MSE) for random {n} businesses: {average_accuracy}")

# display(test_predictions_df.head(5))
# display(predictions_df.head(5))


In [76]:
print('Total Num of Test Predictions:',len(test_predictions_df))
print('Test Prediction, Num of Vadar >1:',len(test_predictions_df[(test_predictions_df['Predicted_VADAR']<-1)]))
print('Test Prediction, Num of Vadar <-1:',len(test_predictions_df[(test_predictions_df['Predicted_VADAR']>1)]))
print('--------------------')
print(' ')
test_predictions_df.to_csv('test_predictions_PA.csv', index=False)

print('Num of biz IDs used in prediction: ',len(predictions_df['Business_ID'].unique()))
print('Num of skipped biz IDs due to insufficient info: ',n-len(predictions_df['Business_ID'].unique()))
print('Overall Prediction, Num of Vadar >1:',len(predictions_df[(predictions_df['Predicted_VADAR']<-1)]))
print('Overall Prediction, Num of Vadar <-1:',len(predictions_df[(predictions_df['Predicted_VADAR']>1)]))

predictions_df.to_csv('predictions_PA.csv', index=False)

Total Num of Test Predictions: 115417
Test Prediction, Num of Vadar >1: 4011
Test Prediction, Num of Vadar <-1: 11046
--------------------
 
Num of biz IDs used in prediction:  31638
Num of skipped biz IDs due to insufficient info:  2401
Overall Prediction, Num of Vadar >1: 409
Overall Prediction, Num of Vadar <-1: 1826


In [80]:
np.sqrt(0.22451754358525897)/(max(test_predictions_df['Actual_VADAR'])-min(test_predictions_df['Actual_VADAR']))

0.23720105246385834